In [ ]:
!pip install -U redis
!pip install -U langchain-chroma

## Cache Setup and Configuration

In [ ]:
import hashlib
import json
import os
import time
from typing import Optional
import redis
from langchain_chroma import Chroma

## QueryCache Class

In [ ]:
class QueryCache:
    """Two-tier cache: exact string match (Redis) + semantic similarity (ChromaDB)."""

    def __init__(self, embedding_model, redis_host="localhost", redis_port=6379,
                 redis_db=0, redis_password=None, cache_dir="./cache",
                 semantic_threshold=0.92, ttl_seconds=86400 * 7, key_prefix="rag_cache:"):

        self.embedding_model = embedding_model
        self.cache_dir = cache_dir
        self.semantic_threshold = semantic_threshold
        self.ttl_seconds = ttl_seconds
        self.key_prefix = key_prefix

        # Redis connection
        self.redis = redis.Redis(
            host=redis_host, port=redis_port, db=redis_db,
            password=redis_password, decode_responses=True
        )

        try:
            self.redis.ping()
            print(f"✅ Connected to Redis at {redis_host}:{redis_port}")
        except redis.ConnectionError:
            raise ConnectionError(
                f"❌ Cannot connect to Redis at {redis_host}:{redis_port}. "
                f"Run: sudo systemctl start redis"
            )

        # ChromaDB for semantic similarity
        os.makedirs(cache_dir, exist_ok=True)
        self.semantic_store = Chroma(
            collection_name="query_cache_semantic",
            embedding_function=embedding_model,
            persist_directory=os.path.join(cache_dir, "semantic_db"),
        )

        count = len(self.redis.keys(f"{self.key_prefix}*"))
        print(f"✅ QueryCache initialized — {count} cached entries in Redis")

    @staticmethod
    def _hash_query(query):
        normalized = query.strip().lower()
        return hashlib.sha256(normalized.encode("utf-8")).hexdigest()

    def _redis_key(self, query_hash):
        return f"{self.key_prefix}{query_hash}"

    def get(self, query):
        """Look up cache. Returns dict if hit, None if miss."""
        query_hash = self._hash_query(query)
        redis_key = self._redis_key(query_hash)

        # Tier 1: Exact match in Redis
        cached_data = self.redis.get(redis_key)
        if cached_data:
            result = json.loads(cached_data)
            result["_cache_type"] = "exact"
            print(f"⚡ Cache HIT (exact match)")
            return result

        # Tier 2: Semantic match via ChromaDB
        try:
            results = self.semantic_store.similarity_search_with_relevance_scores(query, k=1)
            if results:
                doc, score = results[0]
                if score >= self.semantic_threshold:
                    similar_hash = doc.metadata.get("query_hash")
                    similar_key = self._redis_key(similar_hash)
                    cached_data = self.redis.get(similar_key)

                    if cached_data:
                        result = json.loads(cached_data)
                        result["_cache_type"] = "semantic"
                        result["_similar_query"] = doc.page_content
                        result["_similarity_score"] = round(score, 4)
                        print(f"⚡ Cache HIT (semantic, score={score:.4f})")
                        print(f"   Matched: \"{doc.page_content}\"")
                        return result
        except Exception:
            pass

        print(f"❌ Cache MISS")
        return None

    def put(self, query, response):
        """Store query-response pair in both Redis and semantic cache."""
        query_hash = self._hash_query(query)
        redis_key = self._redis_key(query_hash)

        clean_response = {k: v for k, v in response.items() if not k.startswith("_")}

        # Store in Redis with automatic TTL expiration
        self.redis.setex(redis_key, self.ttl_seconds, json.dumps(clean_response))

        # Store in ChromaDB for semantic matching
        self.semantic_store.add_texts(
            texts=[query.strip()],
            metadatas=[{"query_hash": query_hash}],
            ids=[query_hash],
        )

        print(f"💾 Cached: \"{query[:60]}\"")

    def clear(self):
        """Flush entire cache."""
        keys = self.redis.keys(f"{self.key_prefix}*")
        if keys:
            self.redis.delete(*keys)

        self.semantic_store.delete_collection()
        self.semantic_store = Chroma(
            collection_name="query_cache_semantic",
            embedding_function=self.embedding_model,
            persist_directory=os.path.join(self.cache_dir, "semantic_db"),
        )
        print("🗑️ Cache cleared")

    def stats(self):
        """Return cache statistics."""
        keys = self.redis.keys(f"{self.key_prefix}*")
        ttls = [self.redis.ttl(k) for k in keys]
        return {
            "total_entries": len(keys),
            "avg_ttl_remaining_hours": round(sum(ttls) / max(len(ttls), 1) / 3600, 1),
        }

## Initialize Cache

In [ ]:
cache = QueryCache(
    embedding_model=local_embeddings,
    redis_host="localhost",
    redis_port=6379,
    cache_dir="./cache",
    semantic_threshold=0.92,
    ttl_seconds=86400 * 7  # 7 days
)

## Test Cache

In [ ]:
# Test store
cache.put("What is operant conditioning?", {"answer": "Test answer", "citations": []})

# Test exact hit
result = cache.get("What is operant conditioning?")
print(f"Exact: {result['_cache_type']}")

# Test semantic hit (paraphrase)
result = cache.get("Explain operant conditioning in psychology")
print(f"Semantic: {result['_cache_type']}, score: {result.get('_similarity_score')}")

# Stats
print(cache.stats())

# Cleanup test data
cache.clear()